###Data-Level Security Using Dynamic Views
This notebokk demonstrates how data-level security can be implemeneted using dynamic views
The implementation covers 2 key capabilities
- Row-Level Security: COntrols which roles are visible to which user
- Column-Level Masking: Controls how senstivie values are displayed

Asimple sales dataset is used to illustrate how different results when queerying the same view.

#### step1: Set up the schema for the table

In [0]:
%sql
use catalog demo;
create schema if not exists data_security;

In [0]:
%sql
USE SCHEMA data_security;


####2. Create the table
A table is defined to represent sales data across multiple regions

In [0]:
%sql
CREATE OR REPLACE TABLE sales_dynamic (
    id INT,
    region STRING,
    email STRING,
    revenue INt
)

####3. Insert Data
The dataset includes records from UK and US Regions

In [0]:
%sql
INSERT INTO sales_dynamic VALUES (1, 'UK', 'john.doe@gmail.com', 10000), (2, 'US', 'jane.smith@yahoo.com', 20000), (3, 'UK', 'bob.jones@hotmail.com', 30000), (4, 'US', 'sara.williams@gmail.com', 40000), (5, 'UK', 'mike.smith@yahoo.com', 50000), (6, 'US', 'lisa.brown@hotmail.com', 60000), (7, 'UK', 'peter.davis@gmail.com', 70000), (8, 'US', 'amy.miller@yahoo.com', 80000);
    
SELECT * FROM sales_dynamic;

####4. Create the Dynamic view
The dynnamic view implements both row-level security and column-level masking

##### Row-Level Security
    - UK group  --> returns only region = 'UK'
    - US group  --> returns only region = 'US'
    - Admin group ---> returns all

##### Column-Level Masking
    - Admin Group --> Full email Visible
    - Other Users --> email is masked

##### Key Function: is_account_group_member()
This function evalueates weather the current iser belongs to a specific group.

Examples:
    - is_account_group_member('uk-sg')
    - is_account_group_member('us-sg')
    - is_account_group_member('admin-sg')

This function returns true or false and is used to derive conditional logic inside the view

In [0]:
%sql
SELECT is_account_group_member('uk-sg')

In [0]:
%sql
CREATE OR REPLACE VIEW  vw_sales_dynamic AS
SELECT 
    id,
    region,
    CASE 
        WHEN is_account_group_member('admin-sg') THEN email
        ELSE CONCAT(SUBSTR(email, 1, 1), '****@', split(email, '@')[1])
    END AS email,
    revenue 
FROM sales_dynamic
WHERE is_account_group_member('admin-sg') OR is_account_group_member('uk-sg') AND region = 'UK'
OR is_account_group_member('us-sg') AND region = 'US';
    


In [0]:
%sql
SELECT * FROM vw_sales_dynamic;

####5. Query the dynamic view